# Fine-tune the URUU Compliance Guidance model

**What this notebook does:** fine-tunes `Qwen2.5-7B-Instruct` (4-bit QLoRA) to reproduce the
`generateComplianceGuidance()` task from URUU's AI Security Assistant
(`src/lib/ai/claude.ts` in the `uruu-platform` repo) — given a tenant's outstanding
compliance requirements plus current security signals, pick the single requirement to
prioritize next and justify it.

**Why this exists:** the original plan was to distill this task from Claude (have Claude
label synthetic examples, fine-tune on those). That's blocked right now — the Anthropic
account backing this platform has zero credit. Rather than wait, this notebook trains on a
**public CVE/CVSS dataset** instead, reformatted into the same JSON shape the real function
uses. CVSS is a real, objective, industry-standard severity score, so every training target
here is grounded in real data — nothing is fabricated or hallucinated.

**Honest limitation:** this teaches the *mechanic* (compare several risk items under a
JSON schema, pick the most urgent, justify it from the evidence given) using vulnerability
data as a stand-in for compliance-requirement data. It won't have the regulatory-domain
nuance (real NDPA/POPIA/ZDPA language) that Claude-distilled data would. Treat this as the
bootstrap model; layer in Claude-distilled data later via
`uruu-platform/scripts/generate-training-data.ts` once the Anthropic account has credit —
that script is already built and tested, just waiting on billing.

**Before running:** attach a CVE/CVSS dataset via Kaggle's "Add Input" panel. Any dataset
with columns roughly like `cve_id`, `description`, `cvss_score` (or `base_score`) works —
cell 2 below auto-detects common column-name variants. Also add your Hugging Face token as
a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets) before running the push-to-hub cell.

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets huggingface_hub

import json
import random
import re
from pathlib import Path

import pandas as pd
import torch

random.seed(42)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Load the CVE/CVSS dataset

Point `CSV_PATH` at whatever file your attached Kaggle dataset provides under
`/kaggle/input/`. Run `!ls /kaggle/input` first if you're not sure of the exact path.

In [ ]:
# List attached datasets to find the right path
!ls /kaggle/input

# EDIT THIS to match your attached dataset's actual CSV filename
CSV_PATH = "/kaggle/input/YOUR-CVE-DATASET-SLUG/YOUR-FILE.csv"

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

In [ ]:
# Auto-detect common column-name variants across different CVE/CVSS Kaggle datasets.
def find_col(candidates, columns):
    lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None

cols = list(df.columns)
col_id = find_col(["cve_id", "cve", "id", "cve id"], cols)
col_desc = find_col(["description", "summary", "desc"], cols)
col_score = find_col(["cvss_score", "base_score", "cvss", "score", "basescore"], cols)
col_severity = find_col(["severity", "cvss_severity", "base_severity"], cols)

assert col_id and col_desc and col_score, (
    f"Could not auto-detect required columns in {cols}. "
    "Edit find_col() candidates above to match your dataset's actual column names."
)
print(f"Using columns -> id: {col_id}, description: {col_desc}, score: {col_score}, severity: {col_severity}")

df = df[[col_id, col_desc, col_score] + ([col_severity] if col_severity else [])].dropna(subset=[col_id, col_desc, col_score])
df[col_score] = pd.to_numeric(df[col_score], errors="coerce")
df = df.dropna(subset=[col_score])
df = df[df[col_score] > 0].reset_index(drop=True)
print(f"{len(df)} usable rows after cleaning")

## 3. Reformat into `generateComplianceGuidance` training examples

These constants are copied verbatim from `src/lib/ai/claude.ts` — the fine-tuned model
must learn the exact same system prompt and user-message format the production API route
(`/api/ai/compliance-guidance`) actually sends at inference time.

In [ ]:
SYSTEM_PROMPT = (
    "You are a compliance advisor for URUU, a cybersecurity SOC platform serving African governments and "
    "critical infrastructure operators. Given a tenant's outstanding (not-yet-compliant) requirements for a "
    "regulatory framework, plus real current security signals for that tenant, recommend which single "
    "requirement to prioritize next and why, with concrete next steps. Base your reasoning strictly on the "
    "data provided — never invent incidents, statistics, or evidence that isn't in the input. "
    "priorityRequirementCode must exactly match one of the provided requirement codes."
)

FRAMEWORKS = [
    {"name": "Nigeria Data Protection Act (NDPA)", "jurisdiction": "Nigeria"},
    {"name": "Protection of Personal Information Act (POPIA)", "jurisdiction": "South Africa"},
    {"name": "Zambia Data Protection Act (ZDPA)", "jurisdiction": "Zambia"},
    {"name": "Kenya Data Protection Act", "jurisdiction": "Kenya"},
    {"name": "Ghana Data Protection Act", "jurisdiction": "Ghana"},
]

OUTSTANDING_STATUSES = ["NOT_STARTED", "IN_PROGRESS", "NON_COMPLIANT"]

# Independent pool for securitySignals.criticalThreats — kept separate from the sampled
# CVEs below, matching production where threats and compliance requirements are distinct
# data sources, not the same records.
THREAT_TITLES = [
    ("Phishing campaign targeting Finance staff credentials", "PHISHING"),
    ("Malware detected on payroll-db-01", "MALWARE"),
    ("Ransomware encryption activity on hr-fileserver", "RANSOMWARE"),
    ("Volumetric DDoS attack against citizen-portal-web", "DDOS"),
    ("Brute-force login attempts against admin-workstation-14", "BRUTE_FORCE"),
    ("SQL injection attempt on tax-processing-api", "SQL_INJECTION"),
    ("Cross-site scripting payload detected on ministry-crm", "XSS"),
    ("Suspected zero-day exploitation on internal-vpn-gateway", "ZERO_DAY"),
]

REMEDIATION_BY_SCORE = {
    "critical": [
        "Apply the vendor-provided patch or mitigation immediately.",
        "Restrict network exposure of the affected system until remediated.",
        "Notify the incident response team and open a tracked remediation ticket.",
        "Verify the fix in staging before deploying to production.",
    ],
    "high": [
        "Schedule the vendor patch for the next maintenance window.",
        "Apply compensating controls (network segmentation, WAF rules) in the interim.",
        "Verify the fix in staging before deploying to production.",
        "Document the remediation in the compliance evidence log.",
    ],
    "default": [
        "Schedule remediation per the standard patch cadence.",
        "Confirm the fix does not require compensating controls in the interim.",
        "Document the remediation in the compliance evidence log.",
    ],
}


def cvss_bucket(score: float) -> str:
    if score >= 9.0:
        return "critical"
    if score >= 7.0:
        return "high"
    if score >= 4.0:
        return "medium"
    return "low"


def short_title(description: str, max_len: int = 70) -> str:
    first_sentence = re.split(r"(?<=[.!?])\s", description.strip())[0]
    return (first_sentence[: max_len - 1] + "…") if len(first_sentence) > max_len else first_sentence


def build_user_content(framework, outstanding, security_signals):
    return (
        f"Framework: {framework['name']} ({framework['jurisdiction']})\n\n"
        f"Outstanding requirements:\n{json.dumps(outstanding, indent=2)}\n\n"
        f"Current security signals for this tenant:\n{json.dumps(security_signals, indent=2)}"
    )


def build_example(rows: pd.DataFrame):
    framework = random.choice(FRAMEWORKS)
    outstanding = []
    for _, row in rows.iterrows():
        outstanding.append({
            "code": str(row[col_id]),
            "title": short_title(str(row[col_desc])),
            "description": str(row[col_desc])[:600],
            "status": random.choice(OUTSTANDING_STATUSES),
            "notes": None if random.random() > 0.4 else "Partial implementation in progress; evidence not yet collected.",
        })

    threat_count = random.randint(0, 3)
    critical_threats = []
    for title, ttype in random.sample(THREAT_TITLES, threat_count) if threat_count else []:
        critical_threats.append({"title": title, "severity": random.choice(["HIGH", "CRITICAL"]), "type": ttype})

    security_signals = {"openIncidents": random.randint(0, 15), "criticalThreats": critical_threats}

    top_row = rows.loc[rows[col_score].idxmax()]
    top_code = str(top_row[col_id])
    top_score = float(top_row[col_score])
    bucket = cvss_bucket(top_score)
    top_status = next(r["status"] for r in outstanding if r["code"] == top_code)

    rationale = (
        f"\"{short_title(str(top_row[col_desc]))}\" ({top_code}) carries the highest severity in this set "
        f"(CVSS {top_score:.1f}, {bucket}), and remains {top_status.replace('_', ' ').lower()}. "
        f"Addressing this first reduces the greatest immediate risk exposure among the outstanding items."
    )
    recommendations = REMEDIATION_BY_SCORE.get(bucket, REMEDIATION_BY_SCORE["default"])

    target = {"priorityRequirementCode": top_code, "rationale": rationale, "recommendations": recommendations}

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_content(framework, outstanding, security_signals)},
            {"role": "assistant", "content": json.dumps(target)},
        ]
    }


N_EXAMPLES = 1500
K_MIN, K_MAX = 3, 8

examples = []
for _ in range(N_EXAMPLES):
    k = random.randint(K_MIN, K_MAX)
    sample = df.sample(n=min(k, len(df)))
    examples.append(build_example(sample))

print(f"Built {len(examples)} training examples")
print(json.dumps(examples[0], indent=2)[:1500])

In [ ]:
OUT_PATH = Path("/kaggle/working/compliance-guidance-training.jsonl")
with OUT_PATH.open("w") as f:
    for ex in examples:
        f.write(json.dumps(ex) + "\n")
print(f"Wrote {len(examples)} examples to {OUT_PATH}")

## 4. Load base model (4-bit QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Fine-tune

T4-friendly settings: small per-device batch, gradient accumulation, 4-bit base weights.
Adjust `num_train_epochs`/`max_steps` if you change `N_EXAMPLES` above.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("json", data_files=str(OUT_PATH), split="train")

def format_example(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

sft_config = SFTConfig(
    output_dir="/kaggle/working/uruu-compliance-guidance-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=2048,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()

## 6. Push the adapter to Hugging Face

Requires an `HF_TOKEN` Kaggle secret (Add-ons → Secrets in the notebook editor) with
write access. Change `HF_REPO` to wherever you want this hosted — keep it **private**,
this is trained on data reformatted from real vulnerability records.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_REPO = "your-hf-username/uruu-compliance-guidance-lora"  # EDIT THIS

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

trainer.model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)
print(f"Pushed adapter to https://huggingface.co/{HF_REPO}")

## 7. Quick sanity check

Run a couple of real-shaped inputs through the fine-tuned model and eyeball the output —
does it return valid JSON, does `priorityRequirementCode` match one of the input codes?

In [ ]:
test_example = build_example(df.sample(n=5))
prompt_messages = test_example["messages"][:2]  # system + user only

inputs = tokenizer.apply_chat_template(prompt_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
output = model.generate(inputs, max_new_tokens=400, temperature=0.3, do_sample=True)
response = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

print("--- Model output ---")
print(response)
print("\n--- Expected shape (from CVSS ground truth) ---")
print(test_example["messages"][2]["content"])